# Function 6 · Week 10 decision review

Recipe quality and a possible second basin

[Detailed observations](observation_review.ipynb) · [Detailed fits](model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 tested an SVR candidate near a strong published peer observation. The archived GP and neural means shown here are comparable replays, not the SVR’s original score.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 6
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_6/Week_09/figures/neural-ne...
Submitted input                  0.445816-0.323600-0.551644-0.755918-0.072632
Forecast units                                                       output y
Saved GP mean                                                       -0.267896
GP latent SD                                                         0.051465
Saved neural mean                                                   -0.316251
Neural seed SD                                                       0.026214
Name: Forecast frozen before the return, dtype: object

## Results

Round 9 remained below the incumbent on x5 = 0. The historical archive has four high returns in a compact interior-x5 region, including −0.00614. That is evidence against a global claim for the face incumbent.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score             -0.241346
Best before return         -0.213960
Best now                   -0.213960
Return rank                 3.000000
Actual in forecast units   -0.241346
Actual minus GP mean        0.026550
Actual minus neural mean    0.074905
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,-0.204982,0.008977,0.213353
1,"2024 archive, including extra starters",-0.006144,0.207816,0.166141


In [3]:
face = official["x5"].round(6).eq(0)
display(official.assign(region=np.where(face, "x5 = 0", "x5 > 0"))
        .groupby("region")["y"].agg(["count", "median", "max"]))
if archive is not None:
    anchor = archive.loc[archive.y.idxmax(), cols].to_numpy(float)
    near = np.linalg.norm(archive[cols].to_numpy() - anchor, axis=1) <= 0.035
    display(archive.loc[near, "y"].agg(["count", "min", "median", "max"])
            .rename("2024 archive near its strongest point"))

,count,median,max
region,,,
x5 = 0,5,-0.328192,-0.213960
x5 > 0,24,-1.301682,-0.241346


count     4.000000
min      -0.064807
median   -0.016893
max      -0.006144
Name: 2024 archive near its strongest point, dtype: float64

## Predictions

**Local peak and global evidence**

The face is a competitive local region in your own observations. Those observations leave the interior understudied. The historical interior basin is a challenge to the official-only model, and transfer testing found mixed prediction gains.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

The revised input is the centroid of four strong archived interior points from one reporting ID. Their outputs range from −0.065 to −0.006. The bounded advisory ranking and the official-data fallback keep x5=0, where the official evidence is concentrated. This is a deliberate transfer test outside the shortlist, with one participant supplying the tight cluster.

**Week 11 depends on the return.** Refine the interior only after a competitive official return. A weak result favours the confirmed face for the next query.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Official GP mean (score),Predictive return SD
Stage,,,
Initial proposal,0.436425-0.308543-0.611598-0.707302-0.000000,-0.233849,0.056020
Advisory experiment,0.496857-0.309192-0.625685-0.791371-0.000000,-0.200175,0.078412
Revised cohort-informed,0.415446-0.370229-0.652604-0.763869-0.109806,-0.290461,0.078237
Official-data fallback,0.477724-0.315120-0.640574-0.732989-0.000000,-0.207383,0.065922


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.11
tolerance = 0.035
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.23, 0.23, 0.22, 0.22, 0.15], noise=[0.05, 1e-05, 0.5],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.11. Score tolerance 0.035 in output y.
Joint posterior over 470 candidates, including 149 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.02148,0.01562,0.10711,0.19738,0.13173,0.17976,6,9,0.04714,2
RBF,0.00000,0.00000,0.14660,0.27741,0.16931,0.24720,6,9,0.04827,0


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.474027-0.261042-0.610550-0.812953-0.000000


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
2,0.477724-0.315120-0.640574-0.732989-0.000000,Local,-0.207383,0.065922,0.029718,0.047478
1,0.491887-0.327955-0.621173-0.727293-0.000000,Local,-0.208804,0.068277,0.029894,0.047346
0,0.474027-0.261042-0.610550-0.812953-0.000000,Local,-0.212293,0.076851,0.031500,0.047262
7,0.458727-0.300452-0.548699-0.758023-0.000000,Local,-0.214288,0.064865,0.025714,0.046082
3,0.462842-0.277563-0.592365-0.762968-0.000000,Local,-0.210838,0.064649,0.027382,0.045692


### Strategy for the remaining queries

Use two candidate regions and a noise-aware acquisition, with one deliberate comparison away from the incumbent. A smaller trust region becomes worthwhile once a competing location returns a promising score.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | The revised route tests the archive interior around x5≈0.11. The official fallback is a local x5=0 point. |
| 11 | Refine the interior only after a competitive official return. A weak result favours the confirmed face for the next query. |
| 12 | Use the remaining evidence to choose between the two regions, allowing one more confirmation when the lead is smaller than the noise. |
| 13 | Exploit the region with the strongest posterior and measured support. |

[Letham et al., noisy expected improvement](https://arxiv.org/abs/1706.07094) · [Eriksson et al., local Bayesian optimisation](https://papers.nips.cc/paper_files/paper/2019/hash/6c990b7aca7bc7058f5e98ea909e924b-Abstract.html)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


3925

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.